Muestra imágenes de a 5, empezando desde la que elijas. Se puede conservar
todas, eliminar todas, o elegir individualmente cuáles conservar (checkbox por imagen).

Usa las funciones de `etl/extract` (leer S3) y `etl/load` (borrar S3).

In [1]:
# Si el notebook está en /notebooks, hay que agregar la raíz del repo
# al path para poder importar los módulos de etl/
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets

from etl.extract.s3_utils import listar_imagenes, descargar_imagen
from etl.load.s3_delete import eliminar_de_s3
from etl.load.s3_copiar import copiar_a_carpeta

In [2]:
BUCKET_NAME = "humanoid-damage-cnn-834257582282-us-east-1-an"
PREFIX = "raw/IDEA/no_damages/"

# Carpeta destino donde se copian las imágenes que se mueven a "processed"
CARPETA_CONSERVADAS = "processed/damage_detection/images/undamaged/"

TAMANO_LOTE = 5

imagenes = listar_imagenes(BUCKET_NAME, PREFIX)
print(f"Se encontraron {len(imagenes)} imágenes en s3://{BUCKET_NAME}/{PREFIX}")

Se encontraron 2688 imágenes en s3://humanoid-damage-cnn-834257582282-us-east-1-an/raw/IDEA/no_damages/


In [3]:
# Pedir desde qué imagen empezar (1 = primera)
inicio = input(f"Hay {len(imagenes)} imagenes. Desde cual queres empezar? (1 a {len(imagenes)}): ")
inicio = int(inicio) - 1 if inicio.strip().isdigit() else 0
inicio = max(0, min(inicio, len(imagenes) - 1))

indice = {"i": inicio}

out = widgets.Output()
zona_checkboxes = widgets.Output()

btn_conservar_todas = widgets.Button(description="Mover a processed", button_style="success")
btn_descartar_todas = widgets.Button(description="Descartar todas", button_style="danger")
btn_elegir = widgets.Button(description="Elegir individualmente", button_style="warning")
btn_confirmar_seleccion = widgets.Button(description="Confirmar seleccion", button_style="info")
btn_confirmar_seleccion.layout.display = "none"

botones_principales = widgets.HBox([btn_conservar_todas, btn_descartar_todas, btn_elegir])

checkboxes = []


def lote_actual():
    return imagenes[indice["i"] : indice["i"] + TAMANO_LOTE]


def mostrar_lote():
    global checkboxes
    checkboxes = []
    btn_confirmar_seleccion.layout.display = "none"

    with out:
        clear_output(wait=True)
        lote = lote_actual()
        if not lote:
            print("Terminaste de revisar todas las imagenes.")
            botones_principales.close()
            return

        total = len(imagenes)
        print(f"Mostrando {indice['i']+1} a {indice['i']+len(lote)} de {total}")

        fig, axes = plt.subplots(1, len(lote), figsize=(3*len(lote), 3))
        if len(lote) == 1:
            axes = [axes]
        for ax, key in zip(axes, lote):
            img = descargar_imagen(BUCKET_NAME, key)
            ax.imshow(img)
            ax.set_title(key.split("/")[-1], fontsize=7)
            ax.axis("off")
        plt.tight_layout()
        plt.show()

    with zona_checkboxes:
        clear_output(wait=True)


def avanzar():
    indice["i"] += TAMANO_LOTE
    mostrar_lote()


def conservar_todas(b):
    for key in lote_actual():
        copiar_a_carpeta(BUCKET_NAME, key, CARPETA_CONSERVADAS)
    avanzar()


def descartar_todas(b):
    # No se toca nada, solo se avanza
    avanzar()


def elegir_individualmente(b):
    global checkboxes
    lote = lote_actual()
    checkboxes = [widgets.Checkbox(value=True, description=key.split("/")[-1], indent=False) for key in lote]

    with zona_checkboxes:
        clear_output(wait=True)
        print("Tildadas = se mueven a processed. Destildadas = se descartan (no se tocan).")
        display(widgets.VBox(checkboxes))

    btn_confirmar_seleccion.layout.display = "inline-block"


def confirmar_seleccion(b):
    lote = lote_actual()
    for key, chk in zip(lote, checkboxes):
        if chk.value:
            copiar_a_carpeta(BUCKET_NAME, key, CARPETA_CONSERVADAS)
    avanzar()


btn_conservar_todas.on_click(conservar_todas)
btn_descartar_todas.on_click(descartar_todas)
btn_elegir.on_click(elegir_individualmente)
btn_confirmar_seleccion.on_click(confirmar_seleccion)

display(botones_principales, btn_confirmar_seleccion, out, zona_checkboxes)
mostrar_lote()

Hay 2688 imagenes. Desde cual queres empezar? (1 a 2688):  2210


Button(button_style='info', description='Confirmar seleccion', layout=Layout(display='none'), style=ButtonStyl…

Output()

Output()